<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">

# The Data Scientist
## Book 4 · Software Engineering, Reproducibility, and Deployment Basics
### Notebook 02 · Notebook to Project Code and Testing
© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br> The Python Quants GmbH | https://tpq.io<br>

https://thedatascientist.dev | https://linktr.ee/dyjh

### Why this notebook exists
Exploratory notebooks are useful, but reusable logic belongs in importable files once
it stabilizes. That is what keeps later work compact.

This cell makes the notebook portable. It finds the book root locally, or clones
`tdscode` in Colab, so the rest of the notebook can use stable paths.

This cell checks the Python version and dependency list so the learner can compare
the runtime with the book requirements.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "4_data"
REPO_URL = "https://github.com/yhilpisch/tdscode"

cwd = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in (cwd, *cwd.parents):
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")
    if not repo_root.exists():
        # Clone the companion repo once in Colab.
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
    BOOK_ROOT = repo_root / BOOK_NAME

os.chdir(BOOK_ROOT)

# Make the book root and the helper folder importable.
for path in (BOOK_ROOT, BOOK_ROOT / "code"):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

requirements = BOOK_ROOT / "requirements.txt"
if requirements.exists() and "google.colab" in sys.modules:
    # Keep Colab aligned with the book dependencies.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )

print("Book root:", BOOK_ROOT)


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
BOOK_ROOT = cwd
for candidate in (cwd, cwd.parent, cwd / "4_data"):
    if (candidate / "src").exists() or (candidate / "code").exists():
        BOOK_ROOT = candidate
        break
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from src import data_checks

print("Project data path:", data_checks.project_data_path())


This cell imports the shared project-health helpers from `src/` so the notebook can
reuse the book logic instead of duplicating it.

In [ ]:
frame = data_checks.load_project_health()
summary = data_checks.project_health_summary(frame)

print(frame.head().to_string(index=False))
print()
print("summary:")
print("\n".join(f"{key}: {value}" for key, value in summary.items()))


This cell loads the project-health data and prints a compact summary. It shows the
raw rows the reusable code works on.

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

tmpdir = Path(tempfile.mkdtemp())
test_file = tmpdir / "test_data_checks_demo.py"
test_file.write_text("\n".join([
    "import sys",
    "from pathlib import Path",
    "",
    "BOOK_ROOT = Path.cwd().resolve()",
    "if not (BOOK_ROOT / \"src\").exists():",
    "    BOOK_ROOT = BOOK_ROOT.parent",
    "sys.path.insert(0, str(BOOK_ROOT))",
    "",
    "from src.data_checks import load_project_health",
    "",
    "def test_expected_columns():",
    "    frame = load_project_health()",
    "    expected = {\"project_id\", \"domain\", \"owner_type\", \"delivery_risk\"}",
    "    assert expected.issubset(frame.columns)",
    "",
]))
env = os.environ.copy()
env["PYTHONPATH"] = str(BOOK_ROOT)
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(test_file)],
    cwd=BOOK_ROOT,
    env=env,
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout.strip())

This cell writes a tiny temporary test file and runs `pytest` against it. The point
is to show how reusable code is checked outside the notebook.

In [ ]:
reusable_steps = [
    "load_project_health",
    "validate_project_health",
    "project_health_summary",
]
print("Reusable steps belong in a module:")
print("\n".join(f"- {step}" for step in reusable_steps))


### Where We Are Heading Next
The next notebook shows how environments, dashboards, and deployment fit around the
same project code.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">